# 05: The honest baseline

**Data.** The 920-row frame rebuilt from the four UCI files, with each patient's hospital
recorded. This frame does not contain the label-decided fill found in notebook 03: missing
values stay missing and are imputed inside the pipeline from training rows only (median /
mode).

**Arms (pre-registered).** A keeps provenance proxies (a missingness indicator per feature).
B drops them, and drops Cholesterol entirely, because a median-imputed zero still marks the
hospital.

**Primary estimate.** Leave-one-source-out: train on three hospitals, test on the fourth.
Discrimination is the n-weighted mean of the four within-hospital AUCs, with a bootstrap
stratified by (hospital, class). That is deviation D6: pooled out-of-fold AUC failed the
shuffled-label gate. Hyperparameters are tuned by an inner leave-one-hospital-out search over
the training hospitals only.

**Control.** 10-fold × 5 repeated stratified CV, pooled out-of-fold AUC per repeat. The
bootstrap interval captures evaluation-sample variance only; repeat-to-repeat spread (training
variance) is reported separately.

Everything comes from `scripts/run_baseline.py` → `results/baseline.json`.

In [1]:
import json
import pandas as pd
from heart_audit.data import PROJECT_ROOT
from heart_audit.plots import honest_estimates

B = json.loads((PROJECT_ROOT / "results" / "baseline.json").read_text(encoding="utf-8"))
NAMES = {"logistic_regression": "Logistic regression", "knn": "k-nearest neighbours", "xgboost": "XGBoost", "mlp": "Small MLP"}
rows = []
for arm in ("A", "B"):
    for m, name in NAMES.items():
        v = B["arms"][arm][m]
        l, c = v["loso"], v["control_10x5"]
        rows.append({"arm": arm, "model": name,
                     "LOSO within-site AUC": l["site_weighted_auc"], "95% CI": f"[{l['ci95'][0]:.3f}, {l['ci95'][1]:.3f}]",
                     "LOSO pooled AUC (biased under null)": l["pooled_auc_biased_under_null"],
                     "10x5 pooled AUC": c["pooled_auc_mean_over_repeats"], "10x5 repeat SD": c["repeat_sd"],
                     "10x5 within-site AUC": c["within_site_auc_repeat1"]})
table = pd.DataFrame(rows).set_index(["arm", "model"])
table.round(3)

LOSO within-site AUC          95% CI  \
arm model                                                        
A   Logistic regression                  0.832  [0.796, 0.865]   
    k-nearest neighbours                 0.816  [0.775, 0.852]   
    XGBoost                              0.823  [0.784, 0.859]   
    Small MLP                            0.803  [0.766, 0.837]   
B   Logistic regression                  0.822  [0.783, 0.857]   
    k-nearest neighbours                 0.823  [0.786, 0.856]   
    XGBoost                              0.826  [0.787, 0.862]   
    Small MLP                            0.758  [0.715, 0.798]   

                          LOSO pooled AUC (biased under null)  \
arm model                                                       
A   Logistic regression                                 0.849   
    k-nearest neighbours                                0.816   
    XGBoost                                             0.818   
    Small MLP                                           0.812   
B   Logistic regression                                 0.810   
    k-nearest neighbours                                0.808   
    XGBoost                                             0.822   
    Small MLP                                           0.766   

                          10x5 pooled AUC  10x5 repeat SD  \
arm model                                                   
A   Logistic regression             0.890           0.001   
    k-nearest neighbours            0.880           0.001   
    XGBoost                         0.877           0.003   
    Small MLP                       0.863           0.006   
B   Logistic regression             0.875           0.001   
    k-nearest neighbours            0.859           0.003   
    XGBoost                         0.867           0.002   
    Small MLP                       0.837           0.013   

                          10x5 within-site AUC  
arm model                                       
A   Logistic regression                  0.829  
    k-nearest neighbours                 0.827  
    XGBoost                              0.811  
    Small MLP                            0.817  
B   Logistic regression                  0.839  
    k-nearest neighbours                 0.826  
    XGBoost                              0.817  
    Small MLP                            0.771

In [2]:
honest_estimates([{"label": f"{r['model']} (arm {r['arm']})", "loso": r["LOSO within-site AUC"],
                   "lo": float(r["95% CI"][1:-1].split(",")[0]), "hi": float(r["95% CI"][1:-1].split(",")[1]),
                   "cv": r["10x5 pooled AUC"]} for r in rows],
                 PROJECT_ROOT / "images" / "honest_estimates.png")

WindowsPath('C:/Users/ethan/Desktop/Coding Projects/heart-disease-audit/images/honest_estimates.png')

![](../images/honest_estimates.png)

Random cross-validation's pooled AUC sits well above the held-out-hospital estimate. Its own
*within-site* AUC does not: it lands near the LOSO value. The extra discrimination in pooled
random CV comes from ranking patients across hospitals whose disease rates range from 36% to
94%. It is not better diagnosis of the patient in front of the model.

## Per hospital (logistic regression)

In [3]:
per = []
for arm in ("A", "B"):
    for site, d in B["arms"][arm]["logistic_regression"]["loso"]["per_site"].items():
        per.append({"arm": arm, "held-out hospital": site, "AUC": d["auc"],
                    "DeLong 95% CI": f"[{d['delong_ci95'][0]:.2f}, {d['delong_ci95'][1]:.2f}]",
                    "diseased": d["n_pos"], "healthy": d["n_neg"], "underpowered": d["underpowered"],
                    "calibration intercept": d["calibration_intercept"], "calibration slope": d["calibration_slope"],
                    "balanced acc. @0.5": d["balanced_accuracy_0.5"], "balanced acc. @train Youden": d["balanced_accuracy_train_youden"]})
pd.DataFrame(per).set_index(["arm", "held-out hospital"]).round(2)

AUC DeLong 95% CI  diseased  healthy  underpowered  \
arm held-out hospital                                                        
A   cleveland          0.86  [0.82, 0.91]       139      164         False   
    hungary            0.89  [0.85, 0.93]       106      188         False   
    switzerland        0.76  [0.55, 0.96]       115        8          True   
    va                 0.75  [0.67, 0.83]       149       51         False   
B   cleveland          0.85  [0.81, 0.89]       139      164         False   
    hungary            0.87  [0.83, 0.92]       106      188         False   
    switzerland        0.75  [0.54, 0.96]       115        8          True   
    va                 0.75  [0.67, 0.83]       149       51         False   

                       calibration intercept  calibration slope  \
arm held-out hospital                                             
A   cleveland                          -0.09               0.97   
    hungary                            -0.84               1.31   
    switzerland                         2.65               0.89   
    va                                  0.45               0.69   
B   cleveland                          -0.51               0.92   
    hungary                            -1.03               2.30   
    switzerland                         2.80               0.87   
    va                                  0.03               0.63   

                       balanced acc. @0.5  balanced acc. @train Youden  
arm held-out hospital                                                   
A   cleveland                        0.77                         0.77  
    hungary                          0.80                         0.82  
    switzerland                      0.68                         0.72  
    va                               0.69                         0.70  
B   cleveland                        0.76                         0.77  
    hungary                          0.74                         0.81  
    switzerland                      0.66                         0.67  
    va                               0.64                         0.63

- **Switzerland is underpowered.** It has 8 healthy patients, so its AUC rests on 8 negatives
  whatever the 115 positives, and its interval is very wide. It is n-weighted into the
  primary estimate as stated, never averaged in silently.
- **Calibration is where the hospital shift shows.** A model trained on the other three
  hospitals badly under-predicts risk at Switzerland (large positive intercept) and
  over-predicts at Hungary. AUC is blind to this, and it is the failure that would matter in
  deployment.
- The threshold is never tuned on the held-out hospital. The Youden threshold comes from each
  training fold's own predictions.
- No interval is given for "performance at a new hospital". With four hospitals, between-site
  variance cannot be estimated. The per-site spread above is descriptive.

## Is anything better than logistic regression? (P8.1)

In [4]:
sig = []
for arm in ("A", "B"):
    for m, s in B["arms"][arm]["significance_vs_lr"].items():
        sig.append({"arm": arm, "model vs logistic regression": NAMES[m], "mean fold AUC difference": s["mean_fold_auc_diff"],
                    "t": s["t"], "p": s["p"], "p (Holm, 3 comparisons)": s["p_holm"]})
display(pd.DataFrame(sig).set_index(["arm", "model vs logistic regression"]).round(4))
B["P8.1"]

mean fold AUC difference       t       p  \
arm model vs logistic regression                                             
A   k-nearest neighbours                           -0.0103 -1.6227  0.1111   
    XGBoost                                        -0.0122 -3.0862  0.0033   
    Small MLP                                      -0.0242 -2.9762  0.0045   
B   k-nearest neighbours                           -0.0150 -2.3857  0.0210   
    XGBoost                                        -0.0068 -1.4412  0.1559   
    Small MLP                                      -0.0377 -2.2987  0.0258   

                                  p (Holm, 3 comparisons)  
arm model vs logistic regression                           
A   k-nearest neighbours                           0.1111  
    XGBoost                                        0.0100  
    Small MLP                                      0.0100  
B   k-nearest neighbours                           0.0629  
    XGBoost                                        0.1559  
    Small MLP                                      0.0629

{'arm_A_p_holm': 0.009994070432052438,
 'arm_B_p_holm': 0.15587467525365886,
 'verdict': 'supported'}

Nadeau–Bengio corrected resampled t-test on the 50 fold-level AUCs of the 10×5 control,
with variance inflated by (1/J + n_test/n_train), J = 50. The spec's "1/10 + 1/9" used
k = 10 for a design with 50 resamples. The code uses J = 50, the standard form for repeated
k-fold. p-values are Holm-corrected across the three comparisons. No model is significantly
better than logistic regression. The only significant differences favour logistic
regression.

## Hyperparameter stability across held-out hospitals (arm A)

In [5]:
stab = {NAMES[m]: {site: ", ".join(f"{k.replace('clf__', '')}={v}" for k, v in d["best_params"].items())
                   for site, d in B["arms"]["A"][m]["loso"]["per_site"].items()} for m in NAMES}
pd.DataFrame(stab).T

,cleveland,hungary,switzerland,va
Logistic regression,C=1.0,C=0.1,C=0.1,C=0.1
k-nearest neighbours,"n_neighbors=51, weights=distance","n_neighbors=51, weights=distance","n_neighbors=51, weights=distance","n_neighbors=15, weights=distance"
XGBoost,"learning_rate=0.05, max_depth=2, n_estimators=100","learning_rate=0.05, max_depth=2, n_estimators=100","learning_rate=0.05, max_depth=2, n_estimators=100","learning_rate=0.05, max_depth=3, n_estimators=100"
Small MLP,"alpha=0.0001, hidden_layer_sizes=(16, 16)","alpha=0.0001, hidden_layer_sizes=(16, 16)","alpha=0.0001, hidden_layer_sizes=(32,)","alpha=0.01, hidden_layer_sizes=(16, 16)"


## Metrics, and what they cannot say

- **AUC** is primary. It depends less on prevalence than accuracy does, but it is not immune:
  case mix differs by hospital too (Switzerland's patients are sicker; VA is a veteran cohort).
- **AUPRC** is excluded on purpose. With prevalence ranging from 36% to 94% across held-out
  hospitals, its own prevalence dependence would add interpretation burden without changing
  any conclusion.
- **External validity.** All four cohorts are patients referred for coronary angiography: a
  selected, high-risk population with strong spectrum and verification bias. Nothing here
  generalises to screening the general population.

## Methodology checks

`tests/test_methodology.py` runs with the suite:
- bootstrap and DeLong 95% AUC intervals cover the true AUC at the nominal rate on a known
  data-generating process;
- the corrected t-test is not anti-conservative (the uncorrected one is);
- over 50 label shuffles, the primary estimate's interval contains 0.5 at the rate the gate
  requires.